# PHẦN VII — POST-DEPLOY VÀ VERIFY OPENSTACK

## Mục tiêu phần này

Sau khi Kolla-Ansible deploy xong, cần kiểm tra OpenStack thực sự hoạt động và tạo tài nguyên dùng chung cho học viên.

Phần này gồm:

1. Kiểm tra OpenStack service sau deploy.
2. Kiểm tra Keystone token.
3. Kiểm tra Nova/Neutron/Cinder/Glance.
4. Kiểm tra tích hợp Ceph backend.
5. Upload image Ubuntu 22.04.
6. Tạo flavor dùng chung.
7. Tạo external network / public network.
8. Tạo 12 project và 12 user học viên.
9. Thiết lập quota cho từng học viên.

## Nơi thực hiện

Thực hiện trên **Controller VM1**:

```bash
ssh ubuntu@10.0.0.11
```

hoặc:

```bash
ssh ubuntu@192.168.150.21
```

## Điều kiện trước khi làm phần này

- PHẦN VI deploy đã hoàn tất.
- File `/etc/kolla/admin-openrc.sh` đã tồn tại.
- `openstack service list` chạy được.
- Các container OpenStack đang chạy.
- Ceph pool `images`, `volumes`, `vms`, `backups` đã có.
- Public/Floating IP range dự kiến:
  ```bash
  192.168.150.200/29
  ```
- Gateway lab:
  ```bash
  192.168.150.254
  ```

## Ghi chú về quyền

Các lệnh tạo public image, flavor, external network, project/user cần quyền admin.

Vì vậy luôn source:

```bash
source /etc/kolla/admin-openrc.sh
```


## BƯỚC 28 — Kiểm tra OpenStack đang chạy

## Mục tiêu

Kiểm tra các thành phần chính:

| Thành phần | Lệnh kiểm tra | Kỳ vọng |
|---|---|---|
| Keystone | `openstack token issue` | Trả về token |
| Service Catalog | `openstack service list` | Có compute/network/image/volume/identity |
| Nova | `openstack compute service list` | nova-compute up |
| Neutron/OVN | `openstack network agent list` | OVN agent alive |
| Cinder | `openstack volume service list` | cinder-volume up |
| Ceph backend | tạo volume test | thấy object trong pool `volumes` |


### BƯỚC 28.1 — Load admin credential

Chạy trên Controller VM1.


In [ ]:
# ============================================================
# BƯỚC 28.1 — Load OpenStack admin credentials
# ============================================================

# Activate Kolla venv nếu cần dùng openstack CLI trong venv
source /opt/kolla-venv/bin/activate

# Load admin-openrc
source /etc/kolla/admin-openrc.sh

# Kiểm tra biến môi trường OpenStack
env | grep OS_

# Kiểm tra openstack command
which openstack
openstack --version


### BƯỚC 28.2 — Kiểm tra Keystone token

Lệnh `openstack token issue` xác nhận:

- Keystone API hoạt động.
- Admin credential đúng.
- Endpoint identity truy cập được.
- Token issue thành công.

Kỳ vọng thấy bảng có các trường như:

- `id`
- `expires`
- `project_id`
- `user_id`


In [ ]:
# ============================================================
# BƯỚC 28.2 — Kiểm tra Keystone token
# ============================================================

source /opt/kolla-venv/bin/activate
source /etc/kolla/admin-openrc.sh

openstack token issue

# Nếu lỗi:
# - Kiểm tra /etc/kolla/admin-openrc.sh
# - Kiểm tra keystone container
# - Kiểm tra endpoint/VIP


### BƯỚC 28.3 — Kiểm tra service catalog

Kỳ vọng có các service chính:

- `keystone` / identity
- `nova` / compute
- `neutron` / network
- `glance` / image
- `cinder` / volume
- `placement`


In [ ]:
# ============================================================
# BƯỚC 28.3 — Kiểm tra OpenStack service list
# ============================================================

source /opt/kolla-venv/bin/activate
source /etc/kolla/admin-openrc.sh

openstack service list

# Kiểm tra endpoint list
openstack endpoint list

# Lọc các service quan trọng
openstack service list | egrep "identity|compute|network|image|volume|placement" || true


### BƯỚC 28.4 — Kiểm tra Nova compute service

Kỳ vọng:

- Có `nova-compute` trên 3 compute node.
- `State = up`
- `Status = enabled`

Nếu compute down, cần kiểm tra container `nova_compute` trên compute node tương ứng.


In [ ]:
# ============================================================
# BƯỚC 28.4 — Kiểm tra Nova compute service
# ============================================================

source /opt/kolla-venv/bin/activate
source /etc/kolla/admin-openrc.sh

openstack compute service list

# Kiểm tra hypervisor
openstack hypervisor list
openstack hypervisor stats show

# Nếu nova-compute down:
# ansible -i /etc/kolla/multinode compute -m shell -a "docker ps | grep nova"
# ansible -i /etc/kolla/multinode compute -m shell -a "docker logs nova_compute --tail 100"


### BƯỚC 28.5 — Kiểm tra Neutron / OVN

Nếu dùng OVN, danh sách agent có thể khác OVS agent truyền thống.

Kỳ vọng:

- OVN metadata/ovn-controller hoặc agent liên quan alive.
- Network service không báo lỗi.


In [ ]:
# ============================================================
# BƯỚC 28.5 — Kiểm tra Neutron/OVN
# ============================================================

source /opt/kolla-venv/bin/activate
source /etc/kolla/admin-openrc.sh

# Network agent list
openstack network agent list || true

# Kiểm tra network service/container nếu cần
docker ps | egrep "neutron|ovn|openvswitch" || true

# Kiểm tra toàn cụm
ansible -i /etc/kolla/multinode all -m shell -a "docker ps --format '{{.Names}}' | egrep 'neutron|ovn|openvswitch' || true"


### BƯỚC 28.6 — Kiểm tra Cinder volume service

Kỳ vọng:

- `cinder-scheduler` up.
- `cinder-volume` up.
- Nếu dùng Ceph, cinder-volume phải kết nối được pool `volumes`.


In [ ]:
# ============================================================
# BƯỚC 28.6 — Kiểm tra Cinder volume service
# ============================================================

source /opt/kolla-venv/bin/activate
source /etc/kolla/admin-openrc.sh

openstack volume service list

# Kiểm tra container cinder
docker ps | grep cinder || true

# Nếu cinder-volume down:
# docker logs cinder_volume --tail 200


### BƯỚC 28.7 — Kiểm tra Glance image service

Glance cần hoạt động trước khi upload image.

Nếu Glance dùng Ceph, upload image sẽ tạo object/RBD trong pool `images`.


In [ ]:
# ============================================================
# BƯỚC 28.7 — Kiểm tra Glance image service
# ============================================================

source /opt/kolla-venv/bin/activate
source /etc/kolla/admin-openrc.sh

openstack image list

# Kiểm tra container glance
docker ps | grep glance || true

# Log nếu cần
# docker logs glance_api --tail 200


### BƯỚC 28.8 — Kiểm tra tích hợp Ceph bằng volume test

Mục tiêu:

1. Tạo volume `test-vol` 1GB từ OpenStack.
2. Kiểm tra volume xuất hiện trong Cinder.
3. Kiểm tra pool Ceph `volumes` có RBD tương ứng.
4. Xóa volume test.

Lưu ý:

- Nếu lệnh `rbd` chưa dùng được trên Controller VM1, cài `ceph-common` hoặc kiểm tra `/etc/ceph/ceph.conf` và keyring.
- Với user `client.cinder`, có thể cần dùng:
  ```bash
  sudo rbd -n client.cinder -p volumes ls
  ```


In [ ]:
# ============================================================
# BƯỚC 28.8 — Kiểm tra Ceph integration bằng Cinder volume
# ============================================================

source /opt/kolla-venv/bin/activate
source /etc/kolla/admin-openrc.sh

# Tạo volume test
openstack volume create --size 1 test-vol

# Đợi volume available
openstack volume list
openstack volume show test-vol

# Kiểm tra trong Ceph pool volumes
# Cách 1: dùng client.cinder
sudo rbd -n client.cinder -p volumes ls || true

# Cách 2: nếu có admin keyring
sudo rbd -p volumes ls || true

# Xóa volume test
openstack volume delete test-vol

# Verify xóa
openstack volume list


### BƯỚC 28.9 — Kiểm tra container toàn cụm

Chạy từ Controller VM1.

Mục tiêu:

- Không có container OpenStack quan trọng bị `Exited`.
- Các service chính chạy trên node đúng role.


In [ ]:
# ============================================================
# BƯỚC 28.9 — Kiểm tra container toàn cụm
# ============================================================

# Container trên Controller VM1
docker ps
docker ps -a --filter status=exited

# Container trên toàn cụm
ansible -i /etc/kolla/multinode all -m shell -a "hostname; docker ps --format '{{.Names}}' | wc -l"

# Container exited trên toàn cụm
ansible -i /etc/kolla/multinode all -m shell -a "hostname; docker ps -a --filter status=exited --format '{{.Names}}'"

# Log nhanh container lỗi nếu có
# docker logs <container_name> --tail 200


## BƯỚC 29 — Tạo tài nguyên shared cho học viên

## Mục tiêu

Tạo tài nguyên dùng chung:

1. Image Ubuntu 22.04 LTS.
2. Flavor:
   - `tiny`
   - `small`
   - `medium`
3. External network:
   - network: `public`
   - subnet: `public-subnet`
   - floating range: `192.168.150.201-192.168.150.206`
   - gateway: `192.168.150.254`

## Lưu ý quan trọng về Floating IP range

Dải:

```bash
192.168.150.200/29
```

bao gồm:

- Network: `192.168.150.200`
- Usable: `192.168.150.201` → `192.168.150.206`
- Broadcast: `192.168.150.207`

Không dùng `.200` làm floating IP cấp cho VM vì `.200` đang được dùng làm external VIP trong `globals.yml`.


### BƯỚC 29.1 — Download Ubuntu 22.04 Cloud Image

Chạy trên Controller VM1.

Image tải về:

```bash
jammy-server-cloudimg-amd64.img
```


In [ ]:
# ============================================================
# BƯỚC 29.1 — Download Ubuntu 22.04 Cloud Image
# ============================================================

source /opt/kolla-venv/bin/activate
source /etc/kolla/admin-openrc.sh

mkdir -p ~/openstack-images
cd ~/openstack-images

wget -c https://cloud-images.ubuntu.com/jammy/current/jammy-server-cloudimg-amd64.img

# Verify image file
ls -lh jammy-server-cloudimg-amd64.img
qemu-img info jammy-server-cloudimg-amd64.img || true


### BƯỚC 29.2 — Upload image lên Glance

Image tạo public để mọi project/học viên dùng được.

Các property:

- `hw_disk_bus=virtio`
- `hw_vif_model=virtio`

giúp VM dùng virtio device.


In [ ]:
# ============================================================
# BƯỚC 29.2 — Upload Ubuntu image lên Glance
# ============================================================

source /opt/kolla-venv/bin/activate
source /etc/kolla/admin-openrc.sh

cd ~/openstack-images

openstack image create \
  --file jammy-server-cloudimg-amd64.img \
  --disk-format qcow2 \
  --container-format bare \
  --public \
  --property hw_disk_bus=virtio \
  --property hw_vif_model=virtio \
  "Ubuntu 22.04 LTS"

# Verify image
openstack image list
openstack image show "Ubuntu 22.04 LTS"


### BƯỚC 29.3 — Verify image đã vào Ceph pool `images`

Nếu Glance backend Ceph hoạt động, pool `images` sẽ có object/RBD tương ứng.


In [ ]:
# ============================================================
# BƯỚC 29.3 — Verify Glance image trong Ceph
# ============================================================

# Kiểm tra bằng OpenStack
openstack image list

# Kiểm tra Ceph pool images
sudo rbd -n client.glance -p images ls || true
sudo rbd -p images ls || true

# Nếu không thấy:
# - kiểm tra glance_backend_ceph trong globals.yml
# - kiểm tra /etc/kolla/config/glance/
# - kiểm tra docker logs glance_api


### BƯỚC 29.4 — Tạo flavors

Tạo 3 flavor cơ bản cho lab:

| Flavor | vCPU | RAM | Disk |
|---|---:|---:|---:|
| `tiny` | 1 | 1024 MB | 10 GB |
| `small` | 2 | 2048 MB | 20 GB |
| `medium` | 4 | 4096 MB | 40 GB |


In [ ]:
# ============================================================
# BƯỚC 29.4 — Tạo flavors
# ============================================================

source /opt/kolla-venv/bin/activate
source /etc/kolla/admin-openrc.sh

openstack flavor create --vcpus 1 --ram 1024 --disk 10 tiny
openstack flavor create --vcpus 2 --ram 2048 --disk 20 small
openstack flavor create --vcpus 4 --ram 4096 --disk 40 medium

# Verify
openstack flavor list


### BƯỚC 29.5 — Tạo external/public network

Network `public` dùng cho Floating IP.

Thông số:

- Provider network type: `flat`
- Physical network: `physnet1`
- External: yes

Lưu ý:

`physnet1` phải khớp với mapping trong Neutron/Kolla. Nếu deploy chưa cấu hình provider mapping, cần kiểm tra lại cấu hình Neutron/OVN.


In [ ]:
# ============================================================
# BƯỚC 29.5 — Tạo external network public
# ============================================================

source /opt/kolla-venv/bin/activate
source /etc/kolla/admin-openrc.sh

openstack network create \
  --external \
  --provider-network-type flat \
  --provider-physical-network physnet1 \
  public

# Verify
openstack network list
openstack network show public


### BƯỚC 29.6 — Tạo public subnet

Subnet:

```bash
192.168.150.200/29
```

Allocation pool:

```bash
192.168.150.201 → 192.168.150.206
```

Gateway:

```bash
192.168.150.254
```

Không bật DHCP trên public subnet.


In [ ]:
# ============================================================
# BƯỚC 29.6 — Tạo public subnet
# ============================================================

source /opt/kolla-venv/bin/activate
source /etc/kolla/admin-openrc.sh

openstack subnet create \
  --network public \
  --subnet-range 192.168.150.200/29 \
  --allocation-pool start=192.168.150.201,end=192.168.150.206 \
  --gateway 192.168.150.254 \
  --no-dhcp \
  public-subnet

# Verify
openstack subnet list
openstack subnet show public-subnet


### BƯỚC 29.7 — Verify tài nguyên shared

Kiểm tra toàn bộ tài nguyên dùng chung đã tạo.


In [ ]:
# ============================================================
# BƯỚC 29.7 — Verify shared resources
# ============================================================

source /opt/kolla-venv/bin/activate
source /etc/kolla/admin-openrc.sh

openstack image list
openstack flavor list
openstack network list
openstack subnet list

# Kiểm tra external network
openstack network show public | egrep "router:external|provider|name|id" || true


## BƯỚC 30 — Tạo Project và User cho 12 học viên

## Mục tiêu

Tạo 12 tenant/project:

```text
project-hv01 ... project-hv12
```

Tạo 12 user:

```text
hv01 ... hv12
```

Gán role:

```text
member
```

Thiết lập quota cho từng project:

| Resource | Quota |
|---|---:|
| cores | 8 |
| RAM | 16384 MB |
| instances | 5 |
| volumes | 5 |
| gigabytes | 100 |
| floating IPs | 3 |
| networks | 3 |
| routers | 2 |

## Password user

Mỗi user có password:

```text
HocVienXX@Lab2025!
```

Ví dụ:

- `hv01`: `HocVien01@Lab2025!`
- `hv12`: `HocVien12@Lab2025!`


### BƯỚC 30.1 — Tạo script `init-tenants.sh`

Chạy trên Controller VM1.


In [ ]:
# ============================================================
# BƯỚC 30.1 — Tạo script init-tenants.sh
# ============================================================

#tenant = project + user + quota + role
#Script này sẽ tạo 12 tenant (project + user) cho học viên, mỗi tenant có quota riêng và role member.
#Tên project sẽ là project-hv01, project-hv02, ..., project-hv12

cat > ~/init-tenants.sh << 'EOF'
#!/bin/bash
set -e

# Load admin credential
source /opt/kolla-venv/bin/activate
source /etc/kolla/admin-openrc.sh

echo "=== Bắt đầu tạo 12 tenant học viên ==="

for i in $(seq -w 01 12); do
  PROJECT="project-hv${i}"
  USER="hv${i}"
  PASS="HocVien${i}@Lab2025!"
  EMAIL="hv${i}@lab.local"

  echo "----------------------------------------"
  echo "Đang xử lý: ${USER} / ${PROJECT}"

  # Tạo Project nếu chưa tồn tại
  if openstack project show "${PROJECT}" >/dev/null 2>&1; then
    echo "Project ${PROJECT} đã tồn tại, bỏ qua tạo mới."
  else
    openstack project create \
      --description "Lab project cho HV${i}" \
      --enable "${PROJECT}"
    echo "✓ Đã tạo project ${PROJECT}"
  fi

  # Tạo User nếu chưa tồn tại
  if openstack user show "${USER}" >/dev/null 2>&1; then
    echo "User ${USER} đã tồn tại, bỏ qua tạo mới."
  else
    openstack user create \
      --password "${PASS}" \
      --email "${EMAIL}" \
      --enable "${USER}"
    echo "✓ Đã tạo user ${USER}"
  fi

  # Gán role member
  openstack role add \
    --project "${PROJECT}" \
    --user "${USER}" member

  echo "✓ Đã gán role member cho ${USER}"

  # Thiết lập quota
  openstack quota set \
    --cores 8 \
    --ram 16384 \
    --instances 5 \
    --volumes 5 \
    --gigabytes 100 \
    --floating-ips 3 \
    --networks 3 \
    --routers 2 \
    "${PROJECT}"

  echo "✓ Đã set quota cho ${PROJECT}"
  echo "✓ Login: ${USER} / ${PASS}"
done

echo "========================================"
echo "=== Danh sách project học viên ==="
openstack project list | grep hv || true

echo "=== Danh sách user học viên ==="
openstack user list | grep hv || true

echo "=== Hoàn tất tạo tenant học viên ==="
EOF

chmod +x ~/init-tenants.sh

ls -lh ~/init-tenants.sh


### BƯỚC 30.2 — Chạy script tạo 12 tenant

Chạy trên Controller VM1.

Script có `set -e`, nếu lỗi sẽ dừng để tránh tạo sai hàng loạt.


In [ ]:
# ============================================================
# BƯỚC 30.2 — Chạy script tạo tenant
# ============================================================

~/init-tenants.sh


### BƯỚC 30.3 — Verify project/user/role

Kiểm tra sau khi chạy script.


In [ ]:
# ============================================================
# BƯỚC 30.3 — Verify tenant/user/role
# ============================================================

source /opt/kolla-venv/bin/activate
source /etc/kolla/admin-openrc.sh

# Project
openstack project list | grep hv

# User
openstack user list | grep hv

# Role assignment một user mẫu
openstack role assignment list --user hv01 --project project-hv01 --names

# Kiểm tra quota project mẫu
openstack quota show project-hv01

# Kiểm tra tất cả quota nhanh
for i in $(seq -w 01 12); do
  echo "===== project-hv${i} ====="
  openstack quota show project-hv${i} | egrep "cores|ram|instances|volumes|gigabytes|floating|networks|routers"
done


### BƯỚC 30.4 — Test login bằng user học viên mẫu

Tạo file openrc cho `hv01` để test.

Lưu ý:
- Domain thường là `Default`.
- Project là `project-hv01`.
- User là `hv01`.


In [ ]:
# ============================================================
# BƯỚC 30.4 — Tạo file openrc test cho hv01
# ============================================================

cat > ~/hv01-openrc.sh << EOF
export OS_AUTH_URL=${OS_AUTH_URL}
export OS_PROJECT_NAME=project-hv01
export OS_USERNAME=hv01
export OS_PASSWORD='HocVien01@Lab2025!'
export OS_USER_DOMAIN_NAME=Default
export OS_PROJECT_DOMAIN_NAME=Default
export OS_IDENTITY_API_VERSION=3
EOF

# Test login hv01
source ~/hv01-openrc.sh

openstack token issue
openstack server list
openstack volume list
openstack network list

# Quay lại admin nếu cần
source /etc/kolla/admin-openrc.sh


## Debug lỗi thường gặp

### 1. `openstack token issue` lỗi

Kiểm tra:

```bash
source /etc/kolla/admin-openrc.sh
env | grep OS_
docker ps | grep keystone
docker logs keystone --tail 100
```

### 2. `openstack compute service list` không thấy compute

Kiểm tra trên compute node:

```bash
ansible -i /etc/kolla/multinode compute -m shell -a "docker ps | grep nova"
ansible -i /etc/kolla/multinode compute -m shell -a "docker logs nova_compute --tail 100"
```

### 3. `cinder-volume` down

Kiểm tra:

```bash
docker logs cinder_volume --tail 200
ls -lh /etc/kolla/config/cinder/cinder-volume/
cat /etc/ceph/ceph.conf
sudo rbd -n client.cinder -p volumes ls
```

### 4. Upload image lỗi

Kiểm tra Glance:

```bash
docker logs glance_api --tail 200
openstack image list
sudo rbd -n client.glance -p images ls
```

### 5. Tạo external network lỗi `physnet1`

Nguyên nhân thường là provider physical network chưa map đúng trong Neutron/OVN.

Kiểm tra:

```bash
grep -R "physnet" /etc/kolla/
docker logs neutron_server --tail 200
docker logs ovn_controller --tail 200
```

### 6. Tạo user/project lỗi role `member`

Kiểm tra role:

```bash
openstack role list
```

Nếu không có `member`, tạo:

```bash
openstack role create member
```

### 7. Floating IP không cấp được

Kiểm tra subnet/pool:

```bash
openstack subnet show public-subnet
openstack floating ip list
openstack network show public
```


## Checklist hoàn thành PHẦN VII

- [ ] `source /etc/kolla/admin-openrc.sh` hoạt động.
- [ ] `openstack token issue` trả về token.
- [ ] `openstack service list` có service chính.
- [ ] `openstack compute service list` có compute up.
- [ ] `openstack volume service list` có cinder-volume up.
- [ ] `openstack network agent list` không lỗi nghiêm trọng.
- [ ] Tạo/xóa volume test thành công.
- [ ] Volume test xuất hiện trong Ceph pool `volumes`.
- [ ] Image Ubuntu 22.04 LTS đã upload.
- [ ] Image nằm trong Ceph pool `images`.
- [ ] Flavor `tiny`, `small`, `medium` đã tạo.
- [ ] External network `public` đã tạo.
- [ ] Public subnet `public-subnet` đã tạo.
- [ ] 12 project `project-hv01` → `project-hv12` đã tạo.
- [ ] 12 user `hv01` → `hv12` đã tạo.
- [ ] Role `member` đã gán cho từng user.
- [ ] Quota đã set cho từng project.
- [ ] Test login `hv01` thành công.
